# Casual Conversations v2 - Download & Process (Colab)

Downloads CCv2 zip parts, extracts **only English** scripted/nonscripted MP4s, converts to 16kHz mono WAV, saves to Google Drive.

**Disk-safe**: Processes one file at a time from each zip — peak disk usage is just the zip + 1 MP4 + 1 WAV. Checks available disk before downloading.

**Session-resume**: If Colab disconnects, re-run all cells. Picks up where it left off.

## Setup
1. Upload your `ccv2_links.txt` file (tab-separated: `filename\turl`) to Colab files panel, OR paste links in the config cell.
2. Run all cells.

In [ ]:
# ============================================================
# CONFIG - Edit these
# ============================================================

# Where to save on Google Drive (relative to /content/drive/MyDrive/)
DRIVE_FOLDER = "casual_conversations"

# Max files per class (scripted / nonscripted)
MAX_PER_CLASS = 5000

# Max zip parts to download (None = all). Set to 2-3 for testing.
MAX_PARTS = None

# Minimum free disk (GB) required before downloading a zip.
# If a zip is larger than (available - MIN_FREE_GB), it gets skipped.
MIN_FREE_DISK_GB = 5

# Path to links file. Upload ccv2_links.txt to Colab files panel,
# or set to None and paste links in LINKS_TEXT below.
LINKS_FILE = "/content/ccv2_links.txt"

# Alternative: paste links directly here (tab-separated, one per line)
# Set LINKS_FILE = None to use this instead.
LINKS_TEXT = """
""".strip()

In [ ]:
# ============================================================
# 1. Mount Google Drive & install deps
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!apt-get -qq install ffmpeg
!pip install -q soundfile tqdm pandas numpy

print("\nDeps installed. Drive mounted.")

In [ ]:
# ============================================================
# 2. Setup paths & resume state
# ============================================================
import os
import re
import json
import shutil
import subprocess
import zipfile
import time
import urllib.request
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.notebook import tqdm

SR = 16000

# All persistent data goes to Drive
OUT_DIR = Path(f"/content/drive/MyDrive/{DRIVE_FOLDER}")
AUDIO_DIR = OUT_DIR / "audio_scripted"
AUDIO_DIR_UNSCRIPTED = OUT_DIR / "audio_nonscripted"
STATE_FILE = OUT_DIR / "_resume_state.json"

# Temp on LOCAL disk (faster I/O than Drive)
ZIP_TEMP = Path("/content/zip_temp")

for d in [OUT_DIR, AUDIO_DIR, AUDIO_DIR_UNSCRIPTED, ZIP_TEMP]:
    d.mkdir(parents=True, exist_ok=True)

# Patterns
ENGLISH_PATTERN = re.compile(r"(\d+)_english_(scripted|nonscripted)_(\d+)\.mp4$", re.IGNORECASE)
ANY_LANG_PATTERN = re.compile(r"(\d+)_([a-zA-Z]+)_(scripted|nonscripted)_(\d+)\.mp4$", re.IGNORECASE)


def load_state():
    if STATE_FILE.exists():
        with open(STATE_FILE) as f:
            return json.load(f)
    return {"completed_zips": []}


def save_state(state):
    with open(STATE_FILE, "w") as f:
        json.dump(state, f, indent=2)


def get_wav_counts():
    s = len(list(AUDIO_DIR.glob("*.wav"))) if AUDIO_DIR.exists() else 0
    n = len(list(AUDIO_DIR_UNSCRIPTED.glob("*.wav"))) if AUDIO_DIR_UNSCRIPTED.exists() else 0
    return s, n


def get_free_disk_gb(path="/content"):
    """Get free disk space in GB."""
    stat = os.statvfs(path)
    return (stat.f_bavail * stat.f_frsize) / 1e9


def get_remote_size_gb(url):
    """Get file size from HTTP Content-Length header."""
    try:
        req = urllib.request.Request(url, method="HEAD")
        with urllib.request.urlopen(req, timeout=30) as resp:
            cl = resp.headers.get("Content-Length")
            if cl:
                return int(cl) / 1e9
    except Exception:
        pass
    return None


state = load_state()
s, n = get_wav_counts()
free = get_free_disk_gb()
print(f"Output dir:      {OUT_DIR}")
print(f"Already on disk: {s} scripted, {n} nonscripted WAVs")
print(f"Completed zips:  {len(state['completed_zips'])}")
print(f"Free disk:       {free:.1f} GB")
if state['completed_zips']:
    print(f"  Last done: {state['completed_zips'][-1]}")

In [ ]:
# ============================================================
# 3. Parse links
# ============================================================

def parse_links(links_path=None, links_text=None):
    lines = []
    if links_path and os.path.exists(links_path):
        with open(links_path) as f:
            lines = f.readlines()
    elif links_text:
        lines = links_text.strip().split("\n")
    else:
        raise ValueError("No links provided! Upload ccv2_links.txt or set LINKS_TEXT.")

    links = {}
    for line in lines:
        line = line.strip()
        if not line or line.startswith("file_name"):
            continue
        parts = line.split("\t")
        if len(parts) >= 2:
            fname = parts[0].strip()
            url = parts[1].strip()
            if fname == "CCv2_samples.zip" or (fname.startswith("CCv2_part_") and fname.endswith(".zip")):
                links[fname] = url
    return links


links = parse_links(
    links_path=LINKS_FILE if LINKS_FILE and os.path.exists(LINKS_FILE) else None,
    links_text=LINKS_TEXT if LINKS_TEXT else None,
)


def sort_key(name):
    if name == "CCv2_samples.zip":
        return -1
    m = re.search(r"part_(\d+)", name)
    return int(m.group(1)) if m else 0


sorted_links = sorted(links.items(), key=lambda x: sort_key(x[0]))
if MAX_PARTS:
    sorted_links = sorted_links[:MAX_PARTS]

print(f"Found {len(sorted_links)} zip links")
for name, _ in sorted_links[:5]:
    done = " [DONE]" if name in state["completed_zips"] else ""
    print(f"  {name}{done}")
if len(sorted_links) > 5:
    print(f"  ... and {len(sorted_links) - 5} more")

In [ ]:
# ============================================================
# 4. Processing functions (disk-safe: one file at a time)
# ============================================================

def mp4_to_wav(mp4_path, wav_path):
    """Convert MP4 to 16kHz mono WAV."""
    try:
        result = subprocess.run(
            ["ffmpeg", "-i", str(mp4_path), "-vn", "-acodec", "pcm_s16le",
             "-ar", str(SR), "-ac", "1", "-y", "-loglevel", "error", str(wav_path)],
            capture_output=True, timeout=120,
        )
        return result.returncode == 0
    except Exception:
        return False


def get_duration(wav_path):
    try:
        return sf.info(str(wav_path)).duration
    except Exception:
        return 0


def process_zip_streaming(zip_path):
    """
    Process a zip one English file at a time:
      extract 1 MP4 -> convert to WAV -> move WAV to Drive -> delete MP4
    Peak disk = zip + 1 MP4 (~100MB) + 1 WAV (~5MB)
    """
    try:
        zf = zipfile.ZipFile(zip_path, "r")
    except (zipfile.BadZipFile, Exception) as e:
        print(f"  ERROR opening zip: {e}")
        return

    all_names = zf.namelist()

    # Quick stats
    lang_counts = Counter()
    for name in all_names:
        match = ANY_LANG_PATTERN.match(Path(name).name)
        if match:
            lang_counts[match.group(2).lower()] += 1

    english_files = [n for n in all_names if ENGLISH_PATTERN.search(Path(n).name)]
    print(f"  Total files: {len(all_names)} | Languages: {dict(lang_counts)}")
    print(f"  English files: {len(english_files)}")

    if not english_files:
        zf.close()
        return

    converted = 0
    skipped = {"limit": 0, "short": 0, "ffmpeg": 0, "exists": 0}
    tmp_mp4 = Path("/content/_tmp_convert.mp4")
    tmp_wav = Path("/content/_tmp_convert.wav")

    for name in tqdm(english_files, desc="  Extract+Convert", leave=False):
        basename = Path(name).name
        match = ENGLISH_PATTERN.match(basename)
        if not match:
            continue

        script_type = match.group(2)
        wav_dir = AUDIO_DIR if script_type == "scripted" else AUDIO_DIR_UNSCRIPTED
        final_wav = wav_dir / (Path(basename).stem + ".wav")

        # Skip if already converted (resume support)
        if final_wav.exists():
            skipped["exists"] += 1
            continue

        # Check class limit
        s, n = get_wav_counts()
        if script_type == "scripted" and s >= MAX_PER_CLASS:
            skipped["limit"] += 1
            continue
        if script_type == "nonscripted" and n >= MAX_PER_CLASS:
            skipped["limit"] += 1
            continue

        # Extract single MP4 to local temp
        try:
            data = zf.read(name)
            with open(tmp_mp4, "wb") as f:
                f.write(data)
        except Exception:
            tmp_mp4.unlink(missing_ok=True)
            continue

        # Convert to WAV
        if not mp4_to_wav(tmp_mp4, tmp_wav):
            skipped["ffmpeg"] += 1
            tmp_mp4.unlink(missing_ok=True)
            tmp_wav.unlink(missing_ok=True)
            continue

        # Check duration
        duration = get_duration(tmp_wav)
        if duration < 3.0:
            skipped["short"] += 1
            tmp_mp4.unlink(missing_ok=True)
            tmp_wav.unlink(missing_ok=True)
            continue

        # Move WAV to Drive
        shutil.move(str(tmp_wav), str(final_wav))
        converted += 1

        # Delete temp MP4 immediately
        tmp_mp4.unlink(missing_ok=True)

    zf.close()
    # Final cleanup
    tmp_mp4.unlink(missing_ok=True)
    tmp_wav.unlink(missing_ok=True)

    s, n = get_wav_counts()
    print(f"  Converted: {converted} | Already had: {skipped['exists']} | "
          f"Limit: {skipped['limit']} | Short: {skipped['short']} | FFmpeg fail: {skipped['ffmpeg']}")
    print(f"  Total on disk: {s} scripted, {n} nonscripted")


print("Functions loaded.")

In [ ]:
# ============================================================
# 5. MAIN LOOP - Download, process, resume-safe
# ============================================================
import time as _time

state = load_state()
total = len(sorted_links)
t0 = _time.time()
skipped_disk = []

for i, (fname, url) in enumerate(sorted_links, start=1):
    # Skip already completed
    if fname in state["completed_zips"]:
        print(f"[{i}/{total}] {fname} - SKIPPED (already done)")
        continue

    # Check if target reached
    s, n = get_wav_counts()
    if s >= MAX_PER_CLASS and n >= MAX_PER_CLASS:
        print(f"\nTARGET REACHED: {s} scripted, {n} nonscripted. Stopping.")
        break

    elapsed = _time.time() - t0
    free = get_free_disk_gb()
    print(f"\n{'='*60}")
    print(f"[{i}/{total}] {fname}")
    print(f"  Progress: {s} scripted, {n} nonscripted | Free disk: {free:.1f} GB | {elapsed/60:.0f}m elapsed")
    print(f"{'='*60}")

    # --- Check remote file size vs available disk ---
    remote_gb = get_remote_size_gb(url)
    if remote_gb is not None:
        print(f"  Remote size: {remote_gb:.1f} GB")
        if remote_gb > (free - MIN_FREE_DISK_GB):
            print(f"  SKIPPING: need {remote_gb:.1f} GB + {MIN_FREE_DISK_GB} GB buffer, only {free:.1f} GB free")
            skipped_disk.append(fname)
            continue
    else:
        print(f"  Could not check remote size, proceeding cautiously...")
        if free < MIN_FREE_DISK_GB + 10:
            print(f"  SKIPPING: only {free:.1f} GB free and can't verify zip size")
            skipped_disk.append(fname)
            continue

    zip_path = ZIP_TEMP / fname

    # --- Download to local disk ---
    if not zip_path.exists() or zip_path.stat().st_size < 1000:
        print(f"  Downloading...")
        !wget -O "{zip_path}" -q --show-progress "{url}"
        if not zip_path.exists() or zip_path.stat().st_size < 1000:
            print(f"  wget failed, trying curl...")
            !curl -L -o "{zip_path}" --progress-bar "{url}"

    if not zip_path.exists() or zip_path.stat().st_size < 1000:
        print(f"  Download FAILED. Skipping.")
        zip_path.unlink(missing_ok=True)
        continue

    size_gb = zip_path.stat().st_size / 1e9
    print(f"  Downloaded: {size_gb:.1f} GB")

    # --- Process: extract + convert one file at a time ---
    process_zip_streaming(zip_path)

    # --- Delete zip immediately ---
    print(f"  Deleting zip ({size_gb:.1f} GB freed)")
    zip_path.unlink(missing_ok=True)

    # --- Mark done & persist ---
    state["completed_zips"].append(fname)
    save_state(state)
    print(f"  DONE. State saved.")

# --- Summary ---
s, n = get_wav_counts()
elapsed = _time.time() - t0
print(f"\n{'='*60}")
print(f"FINISHED: {s} scripted, {n} nonscripted WAVs")
print(f"Total time: {elapsed/3600:.1f} hours")
if skipped_disk:
    print(f"\nSkipped {len(skipped_disk)} zips due to disk space:")
    for name in skipped_disk:
        print(f"  {name}")
    print(f"\nTo process these: free up disk or re-run after completed zips are cleaned.")
print(f"{'='*60}")

In [ ]:
# ============================================================
# 6. Build manifest (saved to Drive)
# ============================================================

rows = []
for wav_dir, script_type, label, label_int in [
    (AUDIO_DIR, "scripted", "read", 1),
    (AUDIO_DIR_UNSCRIPTED, "nonscripted", "spontaneous", 0),
]:
    if not wav_dir.exists():
        continue
    for wav in tqdm(sorted(wav_dir.glob("*.wav")), desc=f"{script_type}"):
        match = re.match(r"(\d+)_english_", wav.name)
        pid = match.group(1) if match else ""
        duration = get_duration(wav)
        if duration < 3.0:
            continue
        rows.append({
            "filepath": str(wav),
            "filename": wav.name,
            "source": "casual_conversations",
            "label": label,
            "label_int": label_int,
            "script_type": script_type,
            "duration_sec": round(duration, 2),
            "speaker_id": pid,
        })

df = pd.DataFrame(rows)
manifest_path = OUT_DIR / "manifest.csv"
df.to_csv(manifest_path, index=False)

print(f"\n{'='*60}")
print(f"MANIFEST: {manifest_path}")
print(f"{'='*60}")
print(f"Total:       {len(df)}")
print(f"Scripted:    {(df['label_int']==1).sum()}")
print(f"Nonscripted: {(df['label_int']==0).sum()}")
if len(df) > 0:
    print(f"Duration:    {df['duration_sec'].sum()/3600:.1f} hours")
print(f"\nSaved to Google Drive: Drive > My Drive > {DRIVE_FOLDER}/")

In [ ]:
# ============================================================
# 7. (Optional) Download as zip from Colab
# ============================================================
# For large datasets, just use Google Drive UI instead.

DOWNLOAD_DIRECTLY = False  # Set True to trigger browser download

if DOWNLOAD_DIRECTLY:
    import shutil
    zip_out = "/content/casual_conversations_wavs"
    print("Zipping WAVs + manifest...")
    shutil.make_archive(zip_out, 'zip', str(OUT_DIR))
    from google.colab import files
    files.download(zip_out + ".zip")
else:
    print("Download from Google Drive UI:")
    print(f"  Drive > My Drive > {DRIVE_FOLDER}/")
    print(f"\nOr set DOWNLOAD_DIRECTLY = True above.")

In [ ]:
# ============================================================
# 8. Cleanup resume state (run ONLY when fully done)
# ============================================================
# Uncomment to remove the resume state file:
# STATE_FILE.unlink(missing_ok=True)
# print("Resume state cleared.")